In [ ]:
!pip install transformers
!pip install pyannote.audio

In [3]:
import torch
#from .autonotebook import tqdm as notebook_tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
from whisperx_numpy2_compatibility import load_align_model, align
from whisperx_numpy2_compatibility.diarize import assign_word_speakers

import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

In [4]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [5]:
LOCAL_MODEL = False

In [6]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [7]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [8]:
HF_TOKEN="XXXXXX"

if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    align_model="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    align_model='jonatasgrosman/wav2vec2-large-xlsr-53-russian'

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(device)

cpu


In [10]:
#Initializing up wisper pipeline
whisper_model_id="openai/whisper-large-v3"
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
whisper_model.to(device)
whisper_processor = AutoProcessor.from_pretrained(whisper_model_id)
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


c:\Projects\speech_recognition\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\volko\.cache\huggingface\hub\models--openai--whisper-large-v3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


In [11]:
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))
#model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

In [49]:
file_name='audio/2407151757656693.1.0.0.mp3'
#script = whisper_pipe(file_name, return_timestamps='word', generate_kwargs={"language": "russian"})
script = whisper_pipe(file_name, return_timestamps=True)
print(script['chunks'])

c:\Projects\speech_recognition\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


[{'timestamp': (0.0, 29.98), 'text': ' ЗВОНОК ТЕЛЕФОНА'}, {'timestamp': (0.0, 8.0), 'text': ' Если у вас остались вопросы или вам необходима консультация специалиста, скажите «Да» или «Нет».'}, {'timestamp': (8.68, 9.0), 'text': ' Да.'}, {'timestamp': (10.52, 13.42), 'text': ' С уважением к вам и к вашему бизнесу. Интерлизинг.'}]


In [51]:
script

{'text': ' ЗВОНОК ТЕЛЕФОНА Если у вас остались вопросы или вам необходима консультация специалиста, скажите «Да» или «Нет». Да. С уважением к вам и к вашему бизнесу. Интерлизинг.',
 'chunks': [{'timestamp': (0.0, 29.98), 'text': ' ЗВОНОК ТЕЛЕФОНА'},
  {'timestamp': (0.0, 8.0),
   'text': ' Если у вас остались вопросы или вам необходима консультация специалиста, скажите «Да» или «Нет».'},
  {'timestamp': (8.68, 9.0), 'text': ' Да.'},
  {'timestamp': (10.52, 13.42),
   'text': ' С уважением к вам и к вашему бизнесу. Интерлизинг.'}]}

In [37]:
diarized = diarization_pipeline(file_name, min_speakers=1, max_speakers=3)

c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_samp

In [40]:
print(diarized)

[ 00:00:02.916 -->  00:00:04.030] A SPEAKER_02
[ 00:00:11.843 -->  00:00:25.022] B SPEAKER_00
[ 00:00:25.360 -->  00:00:28.262] C SPEAKER_00
[ 00:00:29.292 -->  00:00:29.612] D SPEAKER_02
[ 00:00:32.110 -->  00:00:37.560] E SPEAKER_00
[ 00:00:38.775 -->  00:00:39.096] F SPEAKER_01
[ 00:00:40.615 -->  00:00:43.399] G SPEAKER_00


In [42]:
for turn, _, speaker_label in diarized.itertracks(yield_label=True):
    print(f'{turn.start} - {turn.start}: {speaker_label}')


2.91659375 - 2.91659375: SPEAKER_02
11.84346875 - 11.84346875: SPEAKER_00
25.360343750000002 - 25.360343750000002: SPEAKER_00
29.29221875 - 29.29221875: SPEAKER_02
32.11034375 - 32.11034375: SPEAKER_00
38.775968750000004 - 38.775968750000004: SPEAKER_01
40.61534375 - 40.61534375: SPEAKER_00


In [43]:
print(script['chunks'])

[{'text': ' ЗВОНОК', 'timestamp': (29.24, 29.24)}, {'text': ' ТЕЛЕФОНА', 'timestamp': (29.24, 29.24)}, {'text': ' Если', 'timestamp': (59.9, 59.9)}, {'text': ' у', 'timestamp': (59.9, 59.9)}, {'text': ' вас', 'timestamp': (59.9, 59.9)}, {'text': ' остались', 'timestamp': (59.9, 59.9)}, {'text': ' вопросы', 'timestamp': (59.9, 59.9)}, {'text': ' или', 'timestamp': (59.9, 59.9)}, {'text': ' вам', 'timestamp': (59.9, 59.9)}, {'text': ' необходима', 'timestamp': (59.9, 59.9)}, {'text': ' консультация', 'timestamp': (59.9, 59.9)}, {'text': ' специалиста,', 'timestamp': (59.9, 59.9)}, {'text': ' скажите', 'timestamp': (59.9, 59.9)}, {'text': ' «Да»', 'timestamp': (59.9, 59.9)}, {'text': ' или', 'timestamp': (59.9, 59.9)}, {'text': ' «Нет».', 'timestamp': (59.9, 59.9)}, {'text': ' Да.', 'timestamp': (59.9, 59.94)}, {'text': ' С', 'timestamp': (59.94, 59.94)}, {'text': ' уважением', 'timestamp': (59.94, 59.94)}, {'text': ' к', 'timestamp': (59.94, 59.94)}, {'text': ' вам', 'timestamp': (59.94,

In [38]:
# Combine results
speaker_transcription = []
for chunk in script['chunks']:
    start_time, end_time = chunk["timestamp"][0], chunk["timestamp"][1]
    speaker = "Unknown"
    for turn, _, speaker_label in diarized.itertracks(yield_label=True):
        if turn.start <= start_time <= turn.end:
            speaker = speaker_label
            break
    speaker_transcription.append({
        "start": start_time,
        "end": end_time,
        "speaker": speaker,
        "text": chunk["text"]
    })

In [39]:
speaker_transcription

[{'start': 29.24, 'end': 29.24, 'speaker': 'Unknown', 'text': ' ЗВОНОК'},
 {'start': 29.24, 'end': 29.24, 'speaker': 'Unknown', 'text': ' ТЕЛЕФОНА'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' Если'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' у'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' вас'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' остались'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' вопросы'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' или'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' вам'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' необходима'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' консультация'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' специалиста,'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 'text': ' скажите'},
 {'start': 59.9, 'end': 59.9, 'speaker': 'Unknown', 't

In [26]:
def transcript(file_name):
    logging.info('started')
    script = whisper_pipe(file_name, return_timestamps='word', generate_kwargs={"language": "russian"})
    with open('script_2.txt', "w", encoding="utf-8") as f:
        f.write(script["text"])    
    logging.info('loaded')
    diarized = diarization_pipeline(file_name, min_speakers=5, max_speakers=9)
    logging.info(diarized)
    model_a, metadata = load_align_model(language_code='russian', device=device, model_name=align_model)
    script_aligned = align(script["chunks"], model_a, metadata, file_name, device)
    result_segments, word_seg = list(assign_word_speakers(
        diarized, script_aligned    
    ).values())
    print(result_segments)
    result_segments = script["chunks"]
    transcribed = []
    for result_segment in result_segments:
        transcribed.append(
            {
                "start": result_segment["timestamp"][0],
                "end": result_segment["timestamp"][1],
                "text": result_segment["text"],
                "speaker": result_segment["speaker"] if 'speaker' in result_segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript_2.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [22]:
audios=["audio/2407151757656693.1.0.0.mp3"]#, "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]

In [27]:
for audio in audios:
        transcript(audio)

INFO:root:started
c:\Projects\speech_recognition\.venv\Lib\site-packages\transformers\models\whisper\generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
INFO:root:loaded
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(
c:\Projects\speech_recognition\.venv\Lib\site-packages\torchaudio\_backend\soundfile_backend.py:71: UserWarning: The MPEG_LAYER_III subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/

KeyError: 'start'